In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Reuse helper to convert timestamp -> timezone-aware datetime (UTC+8) and keep epoch too
def attach_dt(df, ts_col='timestamp'):
    # ensure integer
    df[ts_col] = df[ts_col].astype(int)
    # localize to Asia/Shanghai as instructed
    df['dt'] = pd.to_datetime(df[ts_col], unit='s').dt.tz_localize(tz)
    return df

metric_df = attach_dt(metric_df)
trace_df = attach_dt(trace_df)
log_df = attach_dt(log_df)
error_df = attach_dt(error_df)

# Helper to compute summary for numeric-series grouped by component and metric name
def summarize_numeric(df, name_col, value_col='value'):
    g = df.groupby(['cmdb_id', name_col])
    # count, min/max timestamps, and percentiles (P50,P95,P99) computed on full series per group
    summary = g.agg(
        count_points = (value_col, 'count'),
        min_dt = ('dt', 'min'),
        max_dt = ('dt', 'max'),
        p50 = (value_col, lambda s: np.percentile(s.dropna(), 50) if s.dropna().size>0 else np.nan),
        p95 = (value_col, lambda s: np.percentile(s.dropna(), 95) if s.dropna().size>0 else np.nan),
        p99 = (value_col, lambda s: np.percentile(s.dropna(), 99) if s.dropna().size>0 else np.nan),
    ).reset_index()
    # rename columns to required output names
    summary = summary.rename(columns={'cmdb_id':'component', name_col:name_col})
    # format datetimes compactly (include offset)
    summary['min_timestamp'] = summary['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    summary['max_timestamp'] = summary['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    # round percentiles for compact display
    summary['P50(value)'] = summary['p50'].round(6)
    summary['P95(value)'] = summary['p95'].round(6)
    summary['P99(value)'] = summary['p99'].round(6)
    # select and order columns
    out = summary[['component', name_col, 'count_points', 'min_timestamp', 'max_timestamp', 'P50(value)', 'P95(value)', 'P99(value)']]
    # sort by count desc
    out = out.sort_values('count_points', ascending=False).reset_index(drop=True)
    return out

# 1) metric.csv summary
metric_summary = summarize_numeric(metric_df, 'kpi_name')

# 2) trace.csv summary
trace_summary = summarize_numeric(trace_df, 'trace_name')

# 3) log.csv summary
log_summary = summarize_numeric(log_df, 'log_name')

# 4) error_logs.csv: distinct components with count of error messages and up to 5 example messages per component
# Truncate messages to 120 chars for compactness
def make_error_examples(group, max_examples=5, max_len=120):
    # sort by timestamp asc
    g = group.sort_values('timestamp')
    examples = []
    for _, row in g.head(max_examples).iterrows():
        msg = str(row.get('message', ''))
        if len(msg) > max_len:
            msg = msg[:max_len-3] + '...'
        ts = row['dt'].strftime('%Y-%m-%d %H:%M:%S %z')
        examples.append(f"{ts} | {msg}")
    return "; ".join(examples)

if 'message' not in error_df.columns:
    error_df['message'] = error_df.get('message', '')

err_grp = error_df.groupby('cmdb_id')
error_summary = err_grp.agg(
    error_count = ('message', 'count'),
).reset_index().rename(columns={'cmdb_id':'component'})

# attach examples
examples = err_grp.apply(lambda g: make_error_examples(g)).reset_index().rename(columns={0:'examples'})
error_summary = error_summary.merge(examples, left_on='component', right_on='cmdb_id', how='left').drop(columns=['cmdb_id'])
# sort by count desc and limit to top 20
error_summary = error_summary.sort_values('error_count', ascending=False).reset_index(drop=True)
# keep top 20
error_summary = error_summary[['component', 'error_count', 'examples']].head(20)

# Limit metric/trace/log summaries to top 20 rows each (compact)
metric_summary = metric_summary.head(20)
trace_summary = trace_summary.head(20)
log_summary = log_summary.head(20)

# Display compact summaries (as variables; IPython will render them). They contain headers and <=20 rows each.
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
```python
# Summarize findings in plain English using previously computed summaries.
# Reuse metric_summary, trace_summary, log_summary, error_summary from the kernel state.

# Build compact lists for top items
top_metrics = metric_summary[['component','kpi_name','count_points']].head(5).values.tolist()
top_traces = trace_summary[['component','trace_name','count_points','P50(value)','P95(value)','P99(value)']].head(5).values.tolist()
top_logs = log_summary[['component','log_name','count_points','P50(value)','P95(value)','P99(value)']].head(5).values.tolist()
errors = error_summary[['component','error_count','examples']].head(20).values.tolist()

lines = []
lines.append("Telemetry summary (compact):")
lines.append("")
# Metrics
lines.append("1) Metrics (top series by point count):")
for comp, kpi, cnt in top_metrics:
    lines.append(f" - {comp} | {kpi} | points={int(cnt)}")
lines.append("")
# Traces
lines.append("2) Traces (top series; show P50/P95/P99):")
for comp, tname, cnt, p50, p95, p99 in top_traces:
    lines.append(f" - {comp} | {tname} | points={int(cnt)} | P50={p50} | P95={p95} | P99={p99}")
lines.append("")
# Logs
lines.append("3) Logs (top series; show P50/P95/P99):")
for comp, lname, cnt, p50, p95, p99 in top_logs:
    lines.append(f" - {comp} | {lname} | points={int(cnt)} | P50={p50} | P95={p95} | P99={p99}")
lines.append("")
# Error logs
lines.append("4) Error logs (components with errors, count and up to 5 example messages):")
if len(errors)==0:
    lines.append(" - No error log entries found.")
else:
    for comp, cnt, ex in errors:
        lines.append(f" - {comp} | errors={int(cnt)} | examples: {ex}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       component                                           kpi_name  count_points              min_timestamp              max_timestamp    P50(value)    P95(value)    P99(value)
0       apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  7.000000e+00  7.000000e+00  7.000000e+00
1       apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  8.000000e+00  8.000000e+00  8.000000e+00
2           IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  7.000000e+01  7.450000e+01  7.500000e+01
3       apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  4.000000e-01  4.000000e-01  4.000000e-01
4        Mysql02                            OSLinux-CPU_CPU_CPULoad            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.010000e+00  1.505000e+00  1.750000e+00
5        Mysql02                         OSLinux-CPU_CPU_CPUSysTime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.263718e+00  1.420194e+00  1.987836e+00
6        Mysql02                        OSLinux-CPU_CPU_CPUUserTime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.682200e+00  1.968390e+00  2.030166e+00
7        Mysql02                             OSLinux-CPU_CPU_CPUWio            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.035310e+01  1.811335e+01  2.361145e+01
8        Mysql02                        OSLinux-CPU_CPU_CPUidleutil            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  8.622353e+01  8.900757e+01  8.931060e+01
9           IG01                            OSLinux-CPU_CPU_CPULoad            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.070000e+00  1.305000e+00  1.365000e+00
10          IG01                         OSLinux-CPU_CPU_CPUSysTime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  3.991000e-01  5.175630e-01  5.683520e-01
11          IG01                        OSLinux-CPU_CPU_CPUUserTime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  2.522661e+01  2.533367e+01  2.538327e+01
12      apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  5.000000e-01  5.000000e-01  5.000000e-01
13       Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  0.000000e+00  0.000000e+00  0.000000e+00
14       Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  4.000000e+00  4.000000e+00  4.000000e+00
15       Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  5.023081e+06  5.023893e+06  5.023965e+06
16       Redis02                        OSLinux-CPU_CPU_CPUUserTime            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  2.669450e-01  3.337060e-01  4.119520e-01
17  ServiceTest5                                            app.cnt            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  7.900000e+01  1.470000e+02  1.568000e+02
18  ServiceTest5                                            app.mrt            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.104300e-01  6.241250e-01  1.453395e+00
19  ServiceTest5                                             app.rr            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800  1.000000e+02  1.000000e+02  1.000000e+02,    component                       trace_name  count_points              min_timestamp              max_timestamp  P50(value)   P95(value)   P99(value)
0       IG01         trace.self.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.336696     1.447982     1.713115
1       IG01          trace.self.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.538375     7.002100     9.419600
2       IG01             trace.self.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  981.000000  1068.400000  1097.560000
3       IG01  trace.to_Tomcat01.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.287822     0.526815     1.037874
4       IG01   trace.to_Tomcat01.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.339525     3.087815     8.020955
5       IG01      trace.to_Tomcat01.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  118.000000   143.550000   152.520000
6       IG01  trace.to_Tomcat02.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.317363     0.688808     1.166046
7       IG01   trace.to_Tomcat02.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.220700     3.378147     8.298639
8       IG01      trace.to_Tomcat02.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  117.500000   137.650000   151.780000
9       IG01  trace.to_Tomcat03.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.314188     0.656620     1.366171
10      IG01   trace.to_Tomcat03.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.398175     3.101750     8.377985
11      IG01      trace.to_Tomcat03.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  121.000000   145.300000   148.710000
12      IG01  trace.to_Tomcat04.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.322009     0.595312     1.276244
13      IG01   trace.to_Tomcat04.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.204050     2.609740     7.954732
14      IG01      trace.to_Tomcat04.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  122.500000   144.200000   150.970000
15      IG02         trace.self.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.336472     1.633978     1.817650
16      IG02          trace.self.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.747950     6.932850     9.244030
17      IG02             trace.self.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800  948.000000  1106.200000  1132.140000
18      IG02  trace.to_Tomcat01.duration_mean            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    0.325937     0.653657     1.163577
19      IG02   trace.to_Tomcat01.duration_p95            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800    1.730050     3.112320     6.629954,    component          log_name  count_points              min_timestamp              max_timestamp  P50(value)  P95(value)  P99(value)
0   apache02   log.error_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800       0.000      1.0000      2.0000
1   Tomcat03   log.error_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800       0.000      0.0000      0.0000
2   Tomcat01     log.row_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800     233.000    285.0000    293.6000
3   Tomcat01   log.error_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800       0.000      0.0000      0.0000
4   apache02     log.row_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800     481.000    535.0000    552.2000
5   apache01     log.row_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800     488.000    536.5000    546.7000
6   Tomcat04   log.error_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800       0.000      0.0000      0.0000
7   Tomcat03     log.row_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800     233.000    287.0000    294.8000
8   apache01   log.error_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800       0.000      0.5000      1.0000
9   Tomcat04     log.row_count            31  2021-03-10 10:30:00 +0800  2021-03-10 11:00:00 +0800     244.000    272.0000    275.7000
10  Tomcat02   log.error_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800       0.000      0.0000      0.0000
11  Tomcat02     log.row_count            30  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800     241.500    272.6500    283.9400
12   Mysql01     log.row_count            29  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800       5.000     12.4000     18.3200
13   Mysql01   log.error_count            29  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800       0.000      0.0000      0.0000
14  Tomcat02  log.gc_pause_max            23  2021-03-10 10:30:00 +0800  2021-03-10 10:59:00 +0800       0.060      0.0800      0.1190
15  Tomcat01  log.gc_pause_max            13  2021-03-10 10:30:00 +0800  2021-03-10 10:58:00 +0800       0.070      0.0940      0.0988
16  Tomcat03  log.gc_pause_max            12  2021-03-10 10:32:00 +0800  2021-03-10 10:57:00 +0800       0.070      0.1000      0.1000
17  Tomcat04  log.gc_pause_max            12  2021-03-10 10:30:00 +0800  2021-03-10 10:57:00 +0800       0.065      0.0845      0.0889
18   Mysql02     log.row_count             4  2021-03-10 10:35:00 +0800  2021-03-10 10:50:00 +0800       1.000      1.0000      1.0000
19   Mysql02   log.error_count             4  2021-03-10 10:35:00 +0800  2021-03-10 10:50:00 +0800       0.000      0.0000      0.0000,   component  error_count                                           examples
0  apache02            4  2021-03-10 10:58:36 +0800 | IPAddress "POST /U...
1  apache01            2  2021-03-10 10:57:49 +0800 | IPAddress "POST /U...)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing dataframes from kernel state: metric_df, trace_df, log_df, error_df
# Ensure 'dt' exists and is timezone-aware in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')

def ensure_dt(df, ts_col='timestamp'):
    if 'dt' not in df.columns:
        df[ts_col] = df[ts_col].astype(int)
        df['dt'] = pd.to_datetime(df[ts_col], unit='s').dt.tz_localize(tz)
    else:
        # ensure timezone is Asia/Shanghai if naive
        if df['dt'].dt.tz is None:
            df['dt'] = df['dt'].dt.tz_localize(tz)
    return df

metric_df = ensure_dt(metric_df)
trace_df = ensure_dt(trace_df)
log_df = ensure_dt(log_df)
error_df = ensure_dt(error_df)

# Define inclusive incident window in UTC+8
start = pd.Timestamp('2021-03-10 18:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-10 19:00:00').tz_localize(tz)

# Helper to produce grouped summary for a dataframe given series column name
def window_summary(df, file_name, series_col):
    # filter within window inclusive
    mask = (df['dt'] >= start) & (df['dt'] <= end)
    dfw = df.loc[mask]
    if dfw.empty:
        return pd.DataFrame(columns=['file_name','component','series_name','count_points','min_timestamp','max_timestamp'])
    grp = dfw.groupby(['cmdb_id', series_col], dropna=False).agg(
        count_points = ('dt','count'),
        min_dt = ('dt','min'),
        max_dt = ('dt','max')
    ).reset_index().rename(columns={'cmdb_id':'component', series_col:'series_name'})
    grp['min_timestamp'] = grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    grp['max_timestamp'] = grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    grp['file_name'] = file_name
    out = grp[['file_name','component','series_name','count_points','min_timestamp','max_timestamp']]
    return out

# Summaries per file
metric_win = window_summary(metric_df, 'metric.csv', 'kpi_name')
trace_win  = window_summary(trace_df,  'trace.csv',  'trace_name')
log_win    = window_summary(log_df,    'log.csv',    'log_name')

# For error_logs use series_name = 'error' constant
mask_err = (error_df['dt'] >= start) & (error_df['dt'] <= end)
errw = error_df.loc[mask_err]
if errw.empty:
    error_win = pd.DataFrame(columns=['file_name','component','series_name','count_points','min_timestamp','max_timestamp'])
else:
    err_grp = errw.groupby('cmdb_id', dropna=False).agg(
        count_points = ('dt','count'),
        min_dt = ('dt','min'),
        max_dt = ('dt','max')
    ).reset_index().rename(columns={'cmdb_id':'component'})
    err_grp['series_name'] = 'error'
    err_grp['min_timestamp'] = err_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    err_grp['max_timestamp'] = err_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    err_grp['file_name'] = 'error_logs.csv'
    error_win = err_grp[['file_name','component','series_name','count_points','min_timestamp','max_timestamp']]

# Combine and sort by count_points desc, limit 20
combined = pd.concat([metric_win, trace_win, log_win, error_win], ignore_index=True, sort=False)
if not combined.empty:
    combined = combined.sort_values('count_points', ascending=False).reset_index(drop=True)
combined_top20 = combined.head(20)

# Summary line
total_points = int(combined['count_points'].sum()) if not combined.empty else 0
has_data = bool(total_points > 0)
summary_df = pd.DataFrame([{'total_points': total_points, 'has_data': has_data}])

# Display compact outputs
combined_top20, summary_df

```
Out[2]:
```
```python
# Summarize the incident-window check results in plain English.
# Reuse summary_df from the kernel state if available.
if 'summary_df' in globals():
    total_points = int(summary_df.at[0, 'total_points'])
    has_data = bool(summary_df.at[0, 'has_data'])
else:
    total_points = 0
    has_data = False

summary = (
    "Checked telemetry window 2021-03-10 18:30:00 +0800 to 2021-03-10 19:00:00 +0800. "
    "No telemetry points were found in metric.csv, trace.csv, log.csv, or error_logs.csv. "
    f"total_points={total_points}, has_data={has_data}."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [file_name, component, series_name, count_points, min_timestamp, max_timestamp]
Index: [],    total_points  has_data
0             0     False)```
```